In [ ]:
from jax import random
import pandas as pd
from datetime import datetime, timedelta
from numpyro import distributions as dist
from numpyro import infer
import arviz as az
import pickle
import yaml as yml
import pycountry

from estival.sampling import tools as esamp

from emu_renewal.inputs import BASE_PATH, DATA_PATH, VAR_MAP, get_who_targets, get_hosp_target
from emu_renewal.inputs import get_multivars_country_data, get_row_proportions, get_european_var_props, get_filtered_seroprev
from emu_renewal.process import CosineMultiCurve
from emu_renewal.distributions import GammaDens
from emu_renewal.renew import MultiStrainModel
from emu_renewal.outputs import get_spaghetti, get_spagh_df_from_dict, get_output_dir
from emu_renewal.calibration import StandardCalib
from emu_renewal.targets import StandardDispTarget

In [ ]:
country_inputs = yml.safe_load(open(BASE_PATH / "emu_renewal/inputs.yml", "r"))

In [ ]:
# Analysis selection
message = "FB and Google mobility runs for home device"
mob_analysis_type = "no_mob"
country = "Spain"

# Standard inputs
analysis_start = datetime(2020, 9, 1)
analysis_end = datetime(2021, 6, 15)
proc_update_freq = 7
init_duration = 50
var_target_start_date = datetime(2021, 1, 1)
var_target_end_date = datetime(2021, 4, 1)
seed_duration = 10
analysis_to_data_delay = 14

In [ ]:
# Get targets
cases_target, deaths_target, init_data = get_who_targets(country, analysis_start, analysis_end, init_duration, analysis_to_data_delay)
hosp_target = get_hosp_target(country, analysis_start, analysis_end, analysis_to_data_delay)
seroprev_target = get_filtered_seroprev(country, analysis_start, analysis_end)

# Europe-specific wrangling of variants
strains = ["eu", "alpha"]
var_target = get_european_var_props(country, var_target_start_date, var_target_end_date, strains)
alpha_seed_start = analysis_start + timedelta(country_inputs["alpha_seed_delays"][country])
alpha_seed_times = [alpha_seed_start, alpha_seed_start + timedelta(seed_duration)]
seed_times = [alpha_seed_times]

# Compile targets
seroprev_target_dict = {"seropos": StandardDispTarget(seroprev_target, weight=20.0)} if any(seroprev_target) else {}
all_targets = {
    "weekly_cases": StandardDispTarget(cases_target, weight=20.0),
    "weekly_deaths": StandardDispTarget(deaths_target, weight=20.0),
    "prop_eu": StandardDispTarget(var_target, weight=20.0),
    "occupancy": StandardDispTarget(hosp_target, weight=20.0),
} | seroprev_target_dict

In [ ]:
# Mobility
google_mob = pd.read_csv(DATA_PATH / f"mobility/{pycountry.countries.get(name=country).alpha_2}_mob_data.csv", index_col=0)
google_mob.index = pd.to_datetime(google_mob.index)
google_nonresi_mob = google_mob.loc[:, google_mob.columns!="residential"].mean(axis=1).rolling(7).mean().dropna()
fb_mob = pd.read_csv(DATA_PATH / f"mobility/{pycountry.countries.get(name=country).alpha_2}_fbmob_data.csv", index_col=0)["0"]
fb_mob = 1.0 + fb_mob.rolling(7).mean().dropna()
fb_mob.index = pd.to_datetime(fb_mob.index)
combined_mob_data = pd.concat([google_nonresi_mob, google_nonresi_mob ** 2, fb_mob, fb_mob ** 2], axis=1)
combined_mob_data.columns = ["google_nonresi_linear", "google_nonresi_square", "fb_linear", "fb_square"]
combined_mob_data["no_mob"] = 1.0

In [ ]:
# Define model and fitter
model = MultiStrainModel(
    country_inputs["populations"][country], 
    analysis_start, 
    analysis_end, 
    proc_update_freq, 
    CosineMultiCurve(), 
    GammaDens(), 
    init_duration, 
    init_data, 
    GammaDens(), 
    GammaDens(), 
    strains, 
    "eu", 
    seed_times, 
    200.0,
    combined_mob_data[mob_analysis_type].dropna(),
)

In [ ]:
# Set priors
loaded_priors = yml.safe_load(open(BASE_PATH / "emu_renewal/priors.yml", "r"))
duration_priors = {
    k: dist.TruncatedNormal(v["mean"], v["sd"], low=1.0, high=v["mean"] * 2.5) 
    for k, v in loaded_priors["durations"].items()
}
beta_priors = {
    k: dist.Beta(v["alpha"], v["beta"]) 
    for k, v in loaded_priors["beta"].items()
}
other_priors = {
    "alpha_relinfect": dist.TruncatedNormal(1.25, 0.1, low=1.0, high=1.5),
    "rt_init": dist.Normal(0.0, 0.5),
    "shared_dispersion": dist.HalfNormal(0.5),
}
priors = duration_priors | beta_priors | other_priors

In [ ]:
# Define calibration and calib data
calib = StandardCalib(model, priors, all_targets, proc_dispersion=dist.HalfNormal(0.5))

In [ ]:
# Run calibration
analysis_time = datetime.now().strftime("%Y%m%d_%H%M")
kernel = infer.NUTS(calib.calibration, dense_mass=True, init_strategy=calib.custom_init(radius=0.1))
mcmc = infer.MCMC(kernel, num_chains=4, num_samples=1000, num_warmup=1000)
mcmc.warmup(random.PRNGKey(0), collect_warmup=True, extra_fields=["potential_energy"])
pd.DataFrame(mcmc.get_extra_fields(True)["potential_energy"]).T[50:].plot()

In [ ]:
mcmc.run(random.PRNGKey(0), extra_fields=["potential_energy"])

In [ ]:
# Get and store outputs
out_dir = get_output_dir(country, mob_analysis_type, analysis_time)
idata = az.from_dict(mcmc.get_samples(True))
idata.to_netcdf(out_dir / "idata.nc")
idata_sampled = az.extract(idata, num_samples=50)
sample_params = esamp.xarray_to_sampleiterator(idata_sampled)
spaghetti = get_spagh_df_from_dict(get_spaghetti(calib, sample_params))
spaghetti.to_hdf(out_dir / "spaghetti.h5", key="spaghetti")
updates = pd.DataFrame(sample_params.components["proc"], columns=model.epoch.index_to_dti(model.x_proc_vals)).T
updates.to_hdf(out_dir / "updates.h5", key="updates")
likelihood = pd.DataFrame(mcmc.get_extra_fields(True)["potential_energy"]).T
pickle.dump(priors, open(out_dir / "priors.pkl", "wb"))
likelihood.to_hdf(out_dir / "likelihood.h5", key="likelihood")
for t, target in calib.targets.items():
    target.data.to_hdf(out_dir / f"target_{t}.h5", key=t)
with open(out_dir / "description.txt", "w") as file:
    file.write(message)
pd.Series(model.mobility).to_hdf(out_dir / "mobility.h5", key="mobility")

In [ ]:
likelihood.plot()

In [ ]:
mob_analysis_type

In [ ]:
analysis_time

In [ ]:
az.summary(idata)

In [ ]:
az.plot_trace(idata, compact=False);